# Constraint PC algorithm on ASIA dataset

In [ ]:
!pip install pgmpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 48.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12

In [ ]:
!pip install causal-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.0/193.0 kB 6.9 MB/s eta 0:00:00


### Load the data

In [ ]:
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd

# Load the Asia BIF file
reader = BIFReader("/content/asia.bif")
model = reader.get_model()

# Optional: print network summary
print("Nodes:", model.nodes())
print("Edges:", model.edges())

# Simulate data using the model's CPTs
inference = BayesianModelSampling(model)
simulated_data = inference.forward_sample(size=1000, seed=42)  # 1000 rows

# Save to CSV
simulated_data.to_csv("asia_simulated.csv", index=False)
print("Saved asia_simulated.csv")

Nodes: ['asia', 'tub', 'smoke', 'lung', 'bronc', 'either', 'xray', 'dysp']
Edges: [('asia', 'tub'), ('tub', 'either'), ('smoke', 'lung'), ('smoke', 'bronc'), ('lung', 'either'), ('bronc', 'dysp'), ('either', 'xray'), ('either', 'dysp')]


  0%|          | 0/8 [00:00<?, ?it/s]

Saved asia_simulated.csv


**VISUALIZE WITH TEXT LABELS**

In [ ]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import CIT
import pandas as pd
from causallearn.utils.GraphUtils import GraphUtils
from PIL import Image
import io

# Load simulated CSV
df = pd.read_csv("asia_simulated.csv")

# Store original column names before conversion
node_names = list(df.columns)

# Convert categorical data to numerical codes
for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

data = df.values

# Run PC algorithm on simulated data
indep_test = CIT(data, data_type='discrete')
graph = pc(data, indep_test_func=indep_test, alpha=0.05)

# visualize with text labels
dot = GraphUtils.to_pydot(graph.G, labels=node_names)
png_bytes = dot.create_png()
image = Image.open(io.BytesIO(png_bytes))
image.save("pc_asia_output_labels.png")
image.show()

  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
# Inspect unique values after conversion
for col in df.columns:
    print(f"Column '{col}': {df[col].nunique()} unique values")

Column 'asia': 2 unique values
Column 'tub': 2 unique values
Column 'smoke': 2 unique values
Column 'lung': 2 unique values
Column 'bronc': 2 unique values
Column 'either': 2 unique values
Column 'xray': 2 unique values
Column 'dysp': 2 unique values


In [ ]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import CIT
import pandas as pd
from causallearn.utils.GraphUtils import GraphUtils
from PIL import Image
import io

# Load simulated CSV
df = pd.read_csv("asia_simulated.csv")

# Convert categorical data to numerical codes
for col in df.columns:
    df[col] = df[col].astype('category').cat.codes

data = df.values

# Run PC algorithm on simulated data
indep_test = CIT(data, data_type='discrete')
graph = pc(data, indep_test_func=indep_test, alpha=0.05)

# Visualize
dot = GraphUtils.to_pydot(graph.G)
png_bytes = dot.create_png()
image = Image.open(io.BytesIO(png_bytes))
image.save("pc_asia_output.png")
image.show()

  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
print(df.values)

[[0 0 1 ... 1 1 1]
 [0 0 0 ... 0 0 1]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 1 ... 0 0 1]
 [0 0 1 ... 0 0 0]
 [0 0 1 ... 0 0 1]]


### generate ground truth

In [ ]:
from pgmpy.readwrite import BIFReader

# Load Asia model
reader = BIFReader("/content/asia.bif")
model = reader.get_model()

# Ground truth DAG: list of edges
edges = list(model.edges())
print("Ground Truth DAG edges:")
for edge in edges:
    print(f"{edge[0]} → {edge[1]}")

Ground Truth DAG edges:
asia → tub
tub → either
smoke → lung
smoke → bronc
lung → either
bronc → dysp
either → xray
either → dysp


In [ ]:
# visualize
import pydot
from PIL import Image
import io

# Define Asia DAG edges
edges = [
    ("asia", "tub"),
    ("smoke", "lung"),
    ("smoke", "bronc"),
    ("lung", "either"),
    ("tub", "either"),
    ("either", "xray"),
    ("either", "dysp"),
    ("bronc", "dysp")
]

# Create pydot graph
graph = pydot.Dot(graph_type='digraph')
nodes = set(sum(edges, ()))  # unique nodes

for node in nodes:
    graph.add_node(pydot.Node(node))

for src, dst in edges:
    graph.add_edge(pydot.Edge(src, dst))

# Render to PNG and show
png_bytes = graph.create_png()
image = Image.open(io.BytesIO(png_bytes))
image.save("asia_ground_truth_dag.png")
image.show()

### evaluation

In [ ]:
import numpy as np

# Define node names and their indices
node_names = ['asia', 'smoke', 'lung', 'tub', 'bronc', 'either', 'xray', 'dysp']
node_indices = {name: i for i, name in enumerate(node_names)}

# Ground Truth Graph (Image 1)
# Edges: smoke->lung, smoke->bronc, asia->tub, lung->either, tub->either, either->xray, either->dysp, bronc->dysp
ground_truth = np.zeros((8, 8), dtype=int)
ground_truth[node_indices['smoke']][node_indices['lung']] = 1
ground_truth[node_indices['smoke']][node_indices['bronc']] = 1
ground_truth[node_indices['asia']][node_indices['tub']] = 1
ground_truth[node_indices['lung']][node_indices['either']] = 1
ground_truth[node_indices['tub']][node_indices['either']] = 1
ground_truth[node_indices['either']][node_indices['xray']] = 1
ground_truth[node_indices['either']][node_indices['dysp']] = 1
ground_truth[node_indices['bronc']][node_indices['dysp']] = 1

# Generated Graph (Image 2)
# Edges: tub->either, lung->either, lung->smoke, bronc->smoke, bronc->dysp, either->xray
generated = np.zeros((8, 8), dtype=int)
generated[node_indices['tub']][node_indices['either']] = 1
generated[node_indices['lung']][node_indices['either']] = 1
generated[node_indices['lung']][node_indices['smoke']] = 1
generated[node_indices['bronc']][node_indices['smoke']] = 1
generated[node_indices['bronc']][node_indices['dysp']] = 1
generated[node_indices['either']][node_indices['xray']] = 1

# Calculate Structured Hamming Distance
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

# Calculate precision, recall, F1
def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    true_positives = np.sum(true_edges * est_edges)
    false_positives = np.sum((1 - true_edges) * est_edges)
    false_negatives = np.sum(true_edges * (1 - est_edges))

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, true_positives, false_positives, false_negatives

# Calculate metrics
shd = calculate_shd(ground_truth, generated)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, generated)

print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)
print("\nGenerated Adjacency Matrix:")
print(generated)

print(f"\n=== Results ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

# Show edge differences
print(f"\n=== Edge Analysis ===")
print("Ground Truth Edges:")
for i in range(8):
    for j in range(8):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print("\nGenerated Edges:")
for i in range(8):
    for j in range(8):
        if generated[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print("\nCorrect Edges (True Positives):")
for i in range(8):
    for j in range(8):
        if ground_truth[i][j] == 1 and generated[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

print("\nMissed Edges (False Negatives):")
for i in range(8):
    for j in range(8):
        if ground_truth[i][j] == 1 and generated[i][j] == 0:
            print(f"  {node_names[i]} -> {node_names[j]}")

print("\nIncorrect Edges (False Positives):")
for i in range(8):
    for j in range(8):
        if ground_truth[i][j] == 0 and generated[i][j] == 1:
            print(f"  {node_names[i]} -> {node_names[j]}")

Node order: ['asia', 'smoke', 'lung', 'tub', 'bronc', 'either', 'xray', 'dysp']

Ground Truth Adjacency Matrix:
[[0 0 0 1 0 0 0 0]
 [0 0 1 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 1 1]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

Generated Adjacency Matrix:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 1 0 0 0 1 0 0]
 [0 0 0 0 0 1 0 0]
 [0 1 0 0 0 0 0 1]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

=== Results ===
Structured Hamming Distance: 6
True Positives: 4
False Positives: 2
False Negatives: 4
Precision: 0.667
Recall: 0.500
F1-Score: 0.571

=== Edge Analysis ===
Ground Truth Edges:
  asia -> tub
  smoke -> lung
  smoke -> bronc
  lung -> either
  tub -> either
  bronc -> dysp
  either -> xray
  either -> dysp

Generated Edges:
  lung -> smoke
  lung -> either
  tub -> either
  bronc -> smoke
  bronc -> dysp
  either -> xray

Correct Edges (True Positives):
  lung -> either
  tub -> either
  bronc -> dysp
  either -> xray

Missed Edges

# Constraint PC on CANCER

In [ ]:
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd

# Load the Cancer BIF file
reader = BIFReader("/content/cancer.bif")
model = reader.get_model()

# Optional: print network summary
print("Nodes:", model.nodes())
print("Edges:", model.edges())

# Simulate data using the model's CPTs
inference = BayesianModelSampling(model)
simulated_data = inference.forward_sample(size=1000, seed=42)  # 1000 rows

# Save to CSV
simulated_data.to_csv("cancer_simulated.csv", index=False)
print("Saved cancer_simulated.csv")

Nodes: ['Pollution', 'Smoker', 'Cancer', 'Xray', 'Dyspnoea']
Edges: [('Pollution', 'Cancer'), ('Smoker', 'Cancer'), ('Cancer', 'Xray'), ('Cancer', 'Dyspnoea')]


  0%|          | 0/5 [00:00<?, ?it/s]

Saved cancer_simulated.csv


In [ ]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import CIT
import pandas as pd
from causallearn.utils.GraphUtils import GraphUtils
from PIL import Image
import io

# Load simulated CSV
df_cancer = pd.read_csv("cancer_simulated.csv")

# Store original column names before conversion
node_names = list(df_cancer.columns)

# Convert categorical data to numerical codes
for col in df_cancer.columns:
    df_cancer[col] = df_cancer[col].astype('category').cat.codes

data = df_cancer.values

# Run PC algorithm on simulated data
indep_test = CIT(data, data_type='discrete')
graph = pc(data, indep_test_func=indep_test, alpha=0.05)

# visualize with text labels
dot = GraphUtils.to_pydot(graph.G, labels=node_names)
png_bytes = dot.create_png()
image = Image.open(io.BytesIO(png_bytes))
image.save("pc_cancer_output_labels.png")
image.show()

  0%|          | 0/5 [00:00<?, ?it/s]

### generate ground truth

In [ ]:
from pgmpy.readwrite import BIFReader

reader = BIFReader("/content/cancer.bif")
model = reader.get_model()

# Ground truth DAG: list of edges
edges = list(model.edges())
print("Ground Truth DAG edges:")
for edge in edges:
    print(f"{edge[0]} → {edge[1]}")

Ground Truth DAG edges:
Pollution → Cancer
Smoker → Cancer
Cancer → Xray
Cancer → Dyspnoea


### evaluation

In [ ]:
import numpy as np

# Define node names and their indices for CANCER dataset
node_names = ['Pollution', 'Smoker', 'Cancer', 'Xray', 'Dyspnoea']
node_indices = {name: i for i, name in enumerate(node_names)}

# Ground Truth Graph
# Edges: Pollution→Cancer, Smoker→Cancer, Cancer→Xray, Cancer→Dyspnoea
ground_truth = np.zeros((5, 5), dtype=int)
ground_truth[node_indices['Pollution']][node_indices['Cancer']] = 1
ground_truth[node_indices['Smoker']][node_indices['Cancer']] = 1
ground_truth[node_indices['Cancer']][node_indices['Xray']] = 1
ground_truth[node_indices['Cancer']][node_indices['Dyspnoea']] = 1

# Generated Graph (from the image)
# Edges: Pollution→Cancer, Smoker→Cancer, Xray→Cancer, Dyspnoea→Cancer
generated = np.zeros((5, 5), dtype=int)
generated[node_indices['Pollution']][node_indices['Cancer']] = 1
generated[node_indices['Smoker']][node_indices['Cancer']] = 1
generated[node_indices['Xray']][node_indices['Cancer']] = 1
generated[node_indices['Dyspnoea']][node_indices['Cancer']] = 1

# Calculate Structured Hamming Distance
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

# Calculate precision, recall, F1
def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    true_positives = np.sum(true_edges * est_edges)
    false_positives = np.sum((1 - true_edges) * est_edges)
    false_negatives = np.sum(true_edges * (1 - est_edges))

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, true_positives, false_positives, false_negatives

# Calculate metrics
shd = calculate_shd(ground_truth, generated)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, generated)

print("CANCER Dataset Evaluation")
print("=" * 30)
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)
print("\nGenerated Adjacency Matrix:")
print(generated)

print(f"\n=== Results ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

# Show edge differences
print(f"\n=== Edge Analysis ===")
print("Ground Truth Edges:")
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nGenerated Edges:")
for i in range(5):
    for j in range(5):
        if generated[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nCorrect Edges (True Positives):")
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 1 and generated[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nMissed Edges (False Negatives):")
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 1 and generated[i][j] == 0:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nIncorrect Edges (False Positives):")
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 0 and generated[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

CANCER Dataset Evaluation
Node order: ['Pollution', 'Smoker', 'Cancer', 'Xray', 'Dyspnoea']

Ground Truth Adjacency Matrix:
[[0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 0 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Generated Adjacency Matrix:
[[0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 1 0 0]
 [0 0 1 0 0]]

=== Results ===
Structured Hamming Distance: 4
True Positives: 2
False Positives: 2
False Negatives: 2
Precision: 0.500
Recall: 0.500
F1-Score: 0.500

=== Edge Analysis ===
Ground Truth Edges:
  Pollution → Cancer
  Smoker → Cancer
  Cancer → Xray
  Cancer → Dyspnoea

Generated Edges:
  Pollution → Cancer
  Smoker → Cancer
  Xray → Cancer
  Dyspnoea → Cancer

Correct Edges (True Positives):
  Pollution → Cancer
  Smoker → Cancer

Missed Edges (False Negatives):
  Cancer → Xray
  Cancer → Dyspnoea

Incorrect Edges (False Positives):
  Xray → Cancer
  Dyspnoea → Cancer


# EARTHQUAKE dataset

In [ ]:
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd

# Load the EARTHQUAKE BIF file
reader = BIFReader("/content/earthquake.bif")
model = reader.get_model()

# Optional: print network summary
print("Nodes:", model.nodes())
print("Edges:", model.edges())

# Simulate data using the model's CPTs
inference = BayesianModelSampling(model)
simulated_data = inference.forward_sample(size=1000, seed=42)  # 1000 rows

# save to CSV
simulated_data.to_csv("earthquake_simulated.csv", index=False)
print("Saved earthquake_simulated.csv")

Nodes: ['Burglary', 'Earthquake', 'Alarm', 'JohnCalls', 'MaryCalls']
Edges: [('Burglary', 'Alarm'), ('Earthquake', 'Alarm'), ('Alarm', 'JohnCalls'), ('Alarm', 'MaryCalls')]


  0%|          | 0/5 [00:00<?, ?it/s]

Saved earthquake_simulated.csv


In [ ]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import CIT
import pandas as pd
from causallearn.utils.GraphUtils import GraphUtils
from PIL import Image
import io

# Load simulated CSV
df_cancer = pd.read_csv("earthquake_simulated.csv")

# Store original column names before conversion
node_names = list(df_cancer.columns)

# Convert categorical data to numerical codes
for col in df_cancer.columns:
    df_cancer[col] = df_cancer[col].astype('category').cat.codes

data = df_cancer.values

# Run PC algorithm on simulated data
indep_test = CIT(data, data_type='discrete')
graph = pc(data, indep_test_func=indep_test, alpha=0.05)

# visualize with text labels
dot = GraphUtils.to_pydot(graph.G, labels=node_names)
png_bytes = dot.create_png()
image = Image.open(io.BytesIO(png_bytes))
image.save("pc_earthquake_output_labels.png")
image.show()

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from pgmpy.readwrite import BIFReader

reader = BIFReader("/content/earthquake.bif")
model = reader.get_model()

# Ground truth DAG: list of edges
edges = list(model.edges())
print("Ground Truth DAG edges:")
for edge in edges:
    print(f"{edge[0]} → {edge[1]}")

Ground Truth DAG edges:
Burglary → Alarm
Earthquake → Alarm
Alarm → JohnCalls
Alarm → MaryCalls


In [ ]:
import numpy as np

# Define node names and their indices for EARTHQUAKE dataset
node_names = ['Burglary', 'Earthquake', 'Alarm', 'JohnCalls', 'MaryCalls']
node_indices = {name: i for i, name in enumerate(node_names)}

# Ground Truth Graph
# Edges: Burglary→Alarm, Earthquake→Alarm, Alarm→JohnCalls, Alarm→MaryCalls
ground_truth = np.zeros((5, 5), dtype=int)
ground_truth[node_indices['Burglary']][node_indices['Alarm']] = 1
ground_truth[node_indices['Earthquake']][node_indices['Alarm']] = 1
ground_truth[node_indices['Alarm']][node_indices['JohnCalls']] = 1
ground_truth[node_indices['Alarm']][node_indices['MaryCalls']] = 1

# Generated Graph (from the image)
# Edges: Burglary→Alarm, Earthquake→Alarm, Alarm→JohnCalls, Alarm→MaryCalls
generated = np.zeros((5, 5), dtype=int)
generated[node_indices['Burglary']][node_indices['Alarm']] = 1
generated[node_indices['Earthquake']][node_indices['Alarm']] = 1
generated[node_indices['Alarm']][node_indices['JohnCalls']] = 1
generated[node_indices['Alarm']][node_indices['MaryCalls']] = 1

# Calculate Structured Hamming Distance
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

# Calculate precision, recall, F1
def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    true_positives = np.sum(true_edges * est_edges)
    false_positives = np.sum((1 - true_edges) * est_edges)
    false_negatives = np.sum(true_edges * (1 - est_edges))

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, true_positives, false_positives, false_negatives

# Calculate metrics
shd = calculate_shd(ground_truth, generated)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, generated)

print("EARTHQUAKE Dataset Evaluation")
print("=" * 35)
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)
print("\nGenerated Adjacency Matrix:")
print(generated)

print(f"\n=== Results ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

# Show edge differences
print(f"\n=== Edge Analysis ===")
print("Ground Truth Edges:")
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nGenerated Edges:")
for i in range(5):
    for j in range(5):
        if generated[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nCorrect Edges (True Positives):")
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 1 and generated[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nMissed Edges (False Negatives):")
missed_edges = []
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 1 and generated[i][j] == 0:
            missed_edges.append(f"  {node_names[i]} → {node_names[j]}")
if missed_edges:
    for edge in missed_edges:
        print(edge)
else:
    print("  None")

print("\nIncorrect Edges (False Positives):")
incorrect_edges = []
for i in range(5):
    for j in range(5):
        if ground_truth[i][j] == 0 and generated[i][j] == 1:
            incorrect_edges.append(f"  {node_names[i]} → {node_names[j]}")
if incorrect_edges:
    for edge in incorrect_edges:
        print(edge)
else:
    print("  None")

EARTHQUAKE Dataset Evaluation
Node order: ['Burglary', 'Earthquake', 'Alarm', 'JohnCalls', 'MaryCalls']

Ground Truth Adjacency Matrix:
[[0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 0 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]

Generated Adjacency Matrix:
[[0 0 1 0 0]
 [0 0 1 0 0]
 [0 0 0 1 1]
 [0 0 0 0 0]
 [0 0 0 0 0]]

=== Results ===
Structured Hamming Distance: 0
True Positives: 4
False Positives: 0
False Negatives: 0
Precision: 1.000
Recall: 1.000
F1-Score: 1.000

=== Edge Analysis ===
Ground Truth Edges:
  Burglary → Alarm
  Earthquake → Alarm
  Alarm → JohnCalls
  Alarm → MaryCalls

Generated Edges:
  Burglary → Alarm
  Earthquake → Alarm
  Alarm → JohnCalls
  Alarm → MaryCalls

Correct Edges (True Positives):
  Burglary → Alarm
  Earthquake → Alarm
  Alarm → JohnCalls
  Alarm → MaryCalls

Missed Edges (False Negatives):
  None

Incorrect Edges (False Positives):
  None


# SACHS dataset

In [ ]:
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd

# Load the SACHS BIF file
reader = BIFReader("/content/sachs.bif")
model = reader.get_model()

# Optional: print network summary
print("Nodes:", model.nodes())
print("Edges:", model.edges())

# Simulate data using the model's CPTs
inference = BayesianModelSampling(model)
simulated_data = inference.forward_sample(size=1000, seed=42)  # 1000 rows

# save to CSV
simulated_data.to_csv("sachs_simulated.csv", index=False)
print("Saved sachs_simulated.csv")

Nodes: ['Akt', 'Erk', 'Jnk', 'Mek', 'P38', 'PIP2', 'PIP3', 'PKA', 'PKC', 'Plcg', 'Raf']
Edges: [('Erk', 'Akt'), ('Mek', 'Erk'), ('PIP3', 'PIP2'), ('PKA', 'Akt'), ('PKA', 'Erk'), ('PKA', 'Jnk'), ('PKA', 'Mek'), ('PKA', 'P38'), ('PKA', 'Raf'), ('PKC', 'Jnk'), ('PKC', 'Mek'), ('PKC', 'P38'), ('PKC', 'PKA'), ('PKC', 'Raf'), ('Plcg', 'PIP2'), ('Plcg', 'PIP3'), ('Raf', 'Mek')]


  0%|          | 0/11 [00:00<?, ?it/s]

Saved sachs_simulated.csv


In [ ]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import CIT
import pandas as pd
from causallearn.utils.GraphUtils import GraphUtils
from PIL import Image
import io

# Load simulated CSV
df_cancer = pd.read_csv("sachs_simulated.csv")

# Store original column names before conversion
node_names = list(df_cancer.columns)

# Convert categorical data to numerical codes
for col in df_cancer.columns:
    df_cancer[col] = df_cancer[col].astype('category').cat.codes

data = df_cancer.values

# Run PC algorithm on simulated data
indep_test = CIT(data, data_type='discrete')
graph = pc(data, indep_test_func=indep_test, alpha=0.05)

# visualize with text labels
dot = GraphUtils.to_pydot(graph.G, labels=node_names)
png_bytes = dot.create_png()
image = Image.open(io.BytesIO(png_bytes))
image.save("pc_sachs_output_labels.png")
image.show()

  0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
from pgmpy.readwrite import BIFReader

reader = BIFReader("/content/sachs.bif")
model = reader.get_model()

# Ground truth DAG: list of edges
edges = list(model.edges())
print("Ground Truth DAG edges:")
for edge in edges:
    print(f"{edge[0]} → {edge[1]}")

Ground Truth DAG edges:
Erk → Akt
Mek → Erk
PIP3 → PIP2
PKA → Akt
PKA → Erk
PKA → Jnk
PKA → Mek
PKA → P38
PKA → Raf
PKC → Jnk
PKC → Mek
PKC → P38
PKC → PKA
PKC → Raf
Plcg → PIP2
Plcg → PIP3
Raf → Mek


In [ ]:
import numpy as np

# Define node names and their indices for SACHS dataset
node_names = ['Erk', 'Akt', 'Mek', 'PIP3', 'PIP2', 'PKA', 'Jnk', 'P38', 'Raf', 'PKC', 'Plcg']
node_indices = {name: i for i, name in enumerate(node_names)}

# Ground Truth Graph
# Edges: Erk→Akt, Mek→Erk, PIP3→PIP2, PKA→Akt, PKA→Erk, PKA→Jnk, PKA→Mek, PKA→P38, PKA→Raf, PKC→Jnk, PKC→Mek, PKC→P38, PKC→PKA, PKC→Raf, Plcg→PIP2, Plcg→PIP3, Raf→Mek
ground_truth = np.zeros((11, 11), dtype=int)
ground_truth[node_indices['Erk']][node_indices['Akt']] = 1
ground_truth[node_indices['Mek']][node_indices['Erk']] = 1
ground_truth[node_indices['PIP3']][node_indices['PIP2']] = 1
ground_truth[node_indices['PKA']][node_indices['Akt']] = 1
ground_truth[node_indices['PKA']][node_indices['Erk']] = 1
ground_truth[node_indices['PKA']][node_indices['Jnk']] = 1
ground_truth[node_indices['PKA']][node_indices['Mek']] = 1
ground_truth[node_indices['PKA']][node_indices['P38']] = 1
ground_truth[node_indices['PKA']][node_indices['Raf']] = 1
ground_truth[node_indices['PKC']][node_indices['Jnk']] = 1
ground_truth[node_indices['PKC']][node_indices['Mek']] = 1
ground_truth[node_indices['PKC']][node_indices['P38']] = 1
ground_truth[node_indices['PKC']][node_indices['PKA']] = 1
ground_truth[node_indices['PKC']][node_indices['Raf']] = 1
ground_truth[node_indices['Plcg']][node_indices['PIP2']] = 1
ground_truth[node_indices['Plcg']][node_indices['PIP3']] = 1
ground_truth[node_indices['Raf']][node_indices['Mek']] = 1

# Generated Graph (from the image)
# Edges: P38→PKA, Raf→PKA, PKC→PKA, Raf→Mek, PKC→Mek, PKA→Erk, Mek→Erk, PKC→Akt, Akt→Erk, Plcg→Erk, PIP2→Plcg, PIP2→PIP3
generated = np.zeros((11, 11), dtype=int)
generated[node_indices['P38']][node_indices['PKA']] = 1
generated[node_indices['Raf']][node_indices['PKA']] = 1
generated[node_indices['PKC']][node_indices['PKA']] = 1
generated[node_indices['Raf']][node_indices['Mek']] = 1
generated[node_indices['PKC']][node_indices['Mek']] = 1
generated[node_indices['PKA']][node_indices['Erk']] = 1
generated[node_indices['Mek']][node_indices['Erk']] = 1
generated[node_indices['PKC']][node_indices['Akt']] = 1
generated[node_indices['Akt']][node_indices['Erk']] = 1
generated[node_indices['Plcg']][node_indices['Erk']] = 1
generated[node_indices['PIP2']][node_indices['Plcg']] = 1
generated[node_indices['PIP2']][node_indices['PIP3']] = 1

# Calculate Structured Hamming Distance
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

# Calculate precision, recall, F1
def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    true_positives = np.sum(true_edges * est_edges)
    false_positives = np.sum((1 - true_edges) * est_edges)
    false_negatives = np.sum(true_edges * (1 - est_edges))

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, true_positives, false_positives, false_negatives

# Calculate metrics
shd = calculate_shd(ground_truth, generated)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, generated)

print("SACHS Dataset Evaluation")
print("=" * 30)
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)
print("\nGenerated Adjacency Matrix:")
print(generated)

print(f"\n=== Results ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

# Show edge differences
print(f"\n=== Edge Analysis ===")
print("Ground Truth Edges:")
for i in range(11):
    for j in range(11):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nGenerated Edges:")
for i in range(11):
    for j in range(11):
        if generated[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nCorrect Edges (True Positives):")
correct_edges = []
for i in range(11):
    for j in range(11):
        if ground_truth[i][j] == 1 and generated[i][j] == 1:
            correct_edges.append(f"  {node_names[i]} → {node_names[j]}")
if correct_edges:
    for edge in correct_edges:
        print(edge)
else:
    print("  None")

print("\nMissed Edges (False Negatives):")
missed_edges = []
for i in range(11):
    for j in range(11):
        if ground_truth[i][j] == 1 and generated[i][j] == 0:
            missed_edges.append(f"  {node_names[i]} → {node_names[j]}")
if missed_edges:
    for edge in missed_edges:
        print(edge)
else:
    print("  None")

print("\nIncorrect Edges (False Positives):")
incorrect_edges = []
for i in range(11):
    for j in range(11):
        if ground_truth[i][j] == 0 and generated[i][j] == 1:
            incorrect_edges.append(f"  {node_names[i]} → {node_names[j]}")
if incorrect_edges:
    for edge in incorrect_edges:
        print(edge)
else:
    print("  None")

# Additional analysis
print(f"\n=== Detailed Analysis ===")
print(f"Total edges in ground truth: {np.sum(ground_truth)}")
print(f"Total edges in generated graph: {np.sum(generated)}")
print(f"Edge accuracy: {(tp / np.sum(ground_truth) * 100):.1f}% of true edges found")
print(f"Edge error rate: {(fp / np.sum(generated) * 100):.1f}% of generated edges are incorrect")

# Direction analysis
direction_errors = 0
for i in range(11):
    for j in range(11):
        if ground_truth[i][j] == 1 and generated[j][i] == 1:
            direction_errors += 1
            print(f"Direction error: {node_names[i]} → {node_names[j]} (true) vs {node_names[j]} → {node_names[i]} (generated)")

print(f"\nDirection errors: {direction_errors}")
print(f"Pure edge errors (missing/extra): {shd - 2 * direction_errors}")

SACHS Dataset Evaluation
Node order: ['Erk', 'Akt', 'Mek', 'PIP3', 'PIP2', 'PKA', 'Jnk', 'P38', 'Raf', 'PKC', 'Plcg']

Ground Truth Adjacency Matrix:
[[0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 1 1 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 1 1 1 0 0]
 [0 0 0 1 1 0 0 0 0 0 0]]

Generated Adjacency Matrix:
[[0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0 0]
 [0 1 1 0 0 1 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0]]

=== Results ===
Structured Hamming Distance: 19
True Positives: 5
False Positives: 7
False Negatives: 12
Precision: 0.417
Recall: 0.294
F1-Score: 0.345

=== Edge Analysis ===
Ground Truth Edges:
  Erk → Akt
  Mek → Erk
  PIP3 → PIP2
  PKA → Erk
  PKA → Akt
  PKA → Mek

# SURVEY dataset

In [ ]:
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd

# Load the SACHS BIF file
reader = BIFReader("/content/survey.bif")
model = reader.get_model()

# Optional: print network summary
print("Nodes:", model.nodes())
print("Edges:", model.edges())

# Simulate data using the model's CPTs
inference = BayesianModelSampling(model)
simulated_data = inference.forward_sample(size=1000, seed=42)  # 1000 rows

# save to CSV
simulated_data.to_csv("survey_simulated.csv", index=False)
print("Saved survey_simulated.csv")

Nodes: ['A', 'S', 'E', 'O', 'R', 'T']
Edges: [('A', 'E'), ('S', 'E'), ('E', 'O'), ('E', 'R'), ('O', 'T'), ('R', 'T')]


  0%|          | 0/6 [00:00<?, ?it/s]

Saved survey_simulated.csv


In [ ]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import CIT
import pandas as pd
from causallearn.utils.GraphUtils import GraphUtils
from PIL import Image
import io

# Load simulated CSV
df_cancer = pd.read_csv("survey_simulated.csv")

# Store original column names before conversion
node_names = list(df_cancer.columns)

# Convert categorical data to numerical codes
for col in df_cancer.columns:
    df_cancer[col] = df_cancer[col].astype('category').cat.codes

data = df_cancer.values

# Run PC algorithm on simulated data
indep_test = CIT(data, data_type='discrete')
graph = pc(data, indep_test_func=indep_test, alpha=0.05)

# visualize with text labels
dot = GraphUtils.to_pydot(graph.G, labels=node_names)
png_bytes = dot.create_png()
image = Image.open(io.BytesIO(png_bytes))
image.save("pc_survey_output_labels.png")
image.show()

  0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
from pgmpy.readwrite import BIFReader

reader = BIFReader("/content/survey.bif")
model = reader.get_model()

# Ground truth DAG: list of edges
edges = list(model.edges())
print("Ground Truth DAG edges:")
for edge in edges:
    print(f"{edge[0]} → {edge[1]}")

Ground Truth DAG edges:
A → E
S → E
E → O
E → R
O → T
R → T


In [ ]:
import numpy as np

# Define node names and their indices for SURVEY dataset
node_names = ['A', 'S', 'E', 'O', 'T', 'R']
node_indices = {name: i for i, name in enumerate(node_names)}

# Ground Truth Graph
# Edges: A→E, S→E, E→O, E→R, O→T, R→T
ground_truth = np.zeros((6, 6), dtype=int)
ground_truth[node_indices['A']][node_indices['E']] = 1
ground_truth[node_indices['S']][node_indices['E']] = 1
ground_truth[node_indices['E']][node_indices['O']] = 1
ground_truth[node_indices['E']][node_indices['R']] = 1
ground_truth[node_indices['O']][node_indices['T']] = 1
ground_truth[node_indices['R']][node_indices['T']] = 1

# Generated Graph (from the image)
# Edges: E→R, T→R
generated = np.zeros((6, 6), dtype=int)
generated[node_indices['E']][node_indices['R']] = 1
generated[node_indices['T']][node_indices['R']] = 1

# Calculate Structured Hamming Distance
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

# Calculate precision, recall, F1
def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    true_positives = np.sum(true_edges * est_edges)
    false_positives = np.sum((1 - true_edges) * est_edges)
    false_negatives = np.sum(true_edges * (1 - est_edges))

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, true_positives, false_positives, false_negatives

# Calculate metrics
shd = calculate_shd(ground_truth, generated)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, generated)

print("SURVEY Dataset Evaluation")
print("=" * 30)
print("Node order:", node_names)
print("\nGround Truth Adjacency Matrix:")
print(ground_truth)
print("\nGenerated Adjacency Matrix:")
print(generated)

print(f"\n=== Results ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

# Show edge differences
print(f"\n=== Edge Analysis ===")
print("Ground Truth Edges:")
for i in range(6):
    for j in range(6):
        if ground_truth[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nGenerated Edges:")
for i in range(6):
    for j in range(6):
        if generated[i][j] == 1:
            print(f"  {node_names[i]} → {node_names[j]}")

print("\nCorrect Edges (True Positives):")
correct_edges = []
for i in range(6):
    for j in range(6):
        if ground_truth[i][j] == 1 and generated[i][j] == 1:
            correct_edges.append(f"  {node_names[i]} → {node_names[j]}")
if correct_edges:
    for edge in correct_edges:
        print(edge)
else:
    print("  None")

print("\nMissed Edges (False Negatives):")
missed_edges = []
for i in range(6):
    for j in range(6):
        if ground_truth[i][j] == 1 and generated[i][j] == 0:
            missed_edges.append(f"  {node_names[i]} → {node_names[j]}")
if missed_edges:
    for edge in missed_edges:
        print(edge)
else:
    print("  None")

print("\nIncorrect Edges (False Positives):")
incorrect_edges = []
for i in range(6):
    for j in range(6):
        if ground_truth[i][j] == 0 and generated[i][j] == 1:
            incorrect_edges.append(f"  {node_names[i]} → {node_names[j]}")
if incorrect_edges:
    for edge in incorrect_edges:
        print(edge)
else:
    print("  None")

SURVEY Dataset Evaluation
Node order: ['A', 'S', 'E', 'O', 'T', 'R']

Ground Truth Adjacency Matrix:
[[0 0 1 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 1]
 [0 0 0 0 1 0]
 [0 0 0 0 0 0]
 [0 0 0 0 1 0]]

Generated Adjacency Matrix:
[[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 1]
 [0 0 0 0 0 0]
 [0 0 0 0 0 1]
 [0 0 0 0 0 0]]

=== Results ===
Structured Hamming Distance: 6
True Positives: 1
False Positives: 1
False Negatives: 5
Precision: 0.500
Recall: 0.167
F1-Score: 0.250

=== Edge Analysis ===
Ground Truth Edges:
  A → E
  S → E
  E → O
  E → R
  O → T
  R → T

Generated Edges:
  E → R
  T → R

Correct Edges (True Positives):
  E → R

Missed Edges (False Negatives):
  A → E
  S → E
  E → O
  O → T
  R → T

Incorrect Edges (False Positives):
  T → R


# CHILD dataset - 20 nodes

In [ ]:
from pgmpy.readwrite import BIFReader
from pgmpy.sampling import BayesianModelSampling
import pandas as pd

# Load the SACHS BIF file
reader = BIFReader("/content/child.bif")
model = reader.get_model()

# Optional: print network summary
print("Nodes:", model.nodes())
print("Edges:", model.edges())

# Simulate data using the model's CPTs
inference = BayesianModelSampling(model)
simulated_data = inference.forward_sample(size=2000, seed=42)  # 1000 rows

# save to CSV
simulated_data.to_csv("child_simulated.csv", index=False)
print("Saved child_simulated.csv")

Nodes: ['BirthAsphyxia', 'HypDistrib', 'HypoxiaInO2', 'CO2', 'ChestXray', 'Grunting', 'LVHreport', 'LowerBodyO2', 'RUQO2', 'CO2Report', 'XrayReport', 'Disease', 'GruntingReport', 'Age', 'LVH', 'DuctFlow', 'CardiacMixing', 'LungParench', 'LungFlow', 'Sick']
Edges: [('BirthAsphyxia', 'Disease'), ('HypDistrib', 'LowerBodyO2'), ('HypoxiaInO2', 'LowerBodyO2'), ('HypoxiaInO2', 'RUQO2'), ('CO2', 'CO2Report'), ('ChestXray', 'XrayReport'), ('Grunting', 'GruntingReport'), ('Disease', 'Age'), ('Disease', 'LVH'), ('Disease', 'DuctFlow'), ('Disease', 'CardiacMixing'), ('Disease', 'LungParench'), ('Disease', 'LungFlow'), ('Disease', 'Sick'), ('LVH', 'LVHreport'), ('DuctFlow', 'HypDistrib'), ('CardiacMixing', 'HypDistrib'), ('CardiacMixing', 'HypoxiaInO2'), ('LungParench', 'HypoxiaInO2'), ('LungParench', 'CO2'), ('LungParench', 'ChestXray'), ('LungParench', 'Grunting'), ('LungFlow', 'ChestXray'), ('Sick', 'Grunting'), ('Sick', 'Age')]


  0%|          | 0/20 [00:00<?, ?it/s]

Saved child_simulated.csv


In [ ]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import CIT
import pandas as pd
from causallearn.utils.GraphUtils import GraphUtils
from PIL import Image
import io

# Load simulated CSV
df_cancer = pd.read_csv("child_simulated.csv")

# Store original column names before conversion
node_names = list(df_cancer.columns)

# Convert categorical data to numerical codes
for col in df_cancer.columns:
    df_cancer[col] = df_cancer[col].astype('category').cat.codes

data = df_cancer.values

# Run PC algorithm on simulated data
indep_test = CIT(data, data_type='discrete')
graph = pc(data, indep_test_func=indep_test, alpha=0.05)

# visualize with text labels
dot = GraphUtils.to_pydot(graph.G, labels=node_names)
png_bytes = dot.create_png()
image = Image.open(io.BytesIO(png_bytes))
image.save("pc_child_output_labels.png")
image.show()

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
from pgmpy.readwrite import BIFReader

reader = BIFReader("/content/child.bif")
model = reader.get_model()

# Ground truth DAG: list of edges
edges = list(model.edges())
print("Ground Truth DAG edges:")
for edge in edges:
    print(f"{edge[0]} → {edge[1]}")

Ground Truth DAG edges:
BirthAsphyxia → Disease
HypDistrib → LowerBodyO2
HypoxiaInO2 → LowerBodyO2
HypoxiaInO2 → RUQO2
CO2 → CO2Report
ChestXray → XrayReport
Grunting → GruntingReport
Disease → Age
Disease → LVH
Disease → DuctFlow
Disease → CardiacMixing
Disease → LungParench
Disease → LungFlow
Disease → Sick
LVH → LVHreport
DuctFlow → HypDistrib
CardiacMixing → HypDistrib
CardiacMixing → HypoxiaInO2
LungParench → HypoxiaInO2
LungParench → CO2
LungParench → ChestXray
LungParench → Grunting
LungFlow → ChestXray
Sick → Grunting
Sick → Age


In [ ]:
import numpy as np

# Define node names and their indices for CHILD dataset
node_names = ['BirthAsphyxia', 'HypDistrib', 'HypoxiaInO2', 'LowerBodyO2', 'RUQO2', 'CO2', 'CO2Report',
              'ChestXray', 'XrayReport', 'Grunting', 'GruntingReport', 'Disease', 'Age', 'LVH', 'LVHreport',
              'DuctFlow', 'CardiacMixing', 'LungParench', 'LungFlow', 'Sick']
node_indices = {name: i for i, name in enumerate(node_names)}

# Ground Truth Graph
# Edges from the provided list
ground_truth = np.zeros((20, 20), dtype=int)

# Add all ground truth edges
gt_edges = [
    ('BirthAsphyxia', 'Disease'),
    ('HypDistrib', 'LowerBodyO2'),
    ('HypoxiaInO2', 'LowerBodyO2'),
    ('HypoxiaInO2', 'RUQO2'),
    ('CO2', 'CO2Report'),
    ('ChestXray', 'XrayReport'),
    ('Grunting', 'GruntingReport'),
    ('Disease', 'Age'),
    ('Disease', 'LVH'),
    ('Disease', 'DuctFlow'),
    ('Disease', 'CardiacMixing'),
    ('Disease', 'LungParench'),
    ('Disease', 'LungFlow'),
    ('Disease', 'Sick'),
    ('LVH', 'LVHreport'),
    ('DuctFlow', 'HypDistrib'),
    ('CardiacMixing', 'HypDistrib'),
    ('CardiacMixing', 'HypoxiaInO2'),
    ('LungParench', 'HypoxiaInO2'),
    ('LungParench', 'CO2'),
    ('LungParench', 'ChestXray'),
    ('LungParench', 'Grunting'),
    ('LungFlow', 'ChestXray'),
    ('Sick', 'Grunting'),
    ('Sick', 'Age')
]

for source, target in gt_edges:
    ground_truth[node_indices[source]][node_indices[target]] = 1

# Generated Graph (from the image analysis)
# Based on the visual inspection of the provided DAG image
generated = np.zeros((20, 20), dtype=int)

# Generated edges (you may need to adjust these based on careful inspection of the image)
gen_edges = [
    ('LVHreport', 'LVH'),
    ('HypoxiaInO2', 'RUQO2'),
    ('HypoxiaInO2', 'LowerBodyO2'),
    ('LVH', 'CardiacMixing'),
    ('CardiacMixing', 'DuctFlow'),
    ('CardiacMixing', 'HypDistrib'),
    ('Grunting', 'Sick'),
    ('Grunting', 'LungFlow'),
    ('Grunting', 'GruntingReport'),
    ('Sick', 'Age'),
    ('LungFlow', 'ChestXray'),
    ('Age', 'DuctFlow'),
    ('DuctFlow', 'HypDistrib'),
    ('CO2', 'CO2Report'),
    ('LungParench', 'ChestXray'),
    ('ChestXray', 'XrayReport'),
    ('BirthAsphyxia', 'HypDistrib'),
    ('LowerBodyO2', 'HypDistrib'),
    ('HypDistrib', 'Disease'),
    ('Disease', 'LungParench'),
    ('LungParench', 'CO2'),
    ('LungParench', 'Grunting')
]

for source, target in gen_edges:
    generated[node_indices[source]][node_indices[target]] = 1

# Calculate Structured Hamming Distance
def calculate_shd(true_adj, est_adj):
    return np.sum(np.abs(true_adj - est_adj))

# Calculate precision, recall, F1
def calculate_metrics(true_adj, est_adj):
    true_edges = (true_adj != 0).astype(int)
    est_edges = (est_adj != 0).astype(int)

    true_positives = np.sum(true_edges * est_edges)
    false_positives = np.sum((1 - true_edges) * est_edges)
    false_negatives = np.sum(true_edges * (1 - est_edges))

    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return precision, recall, f1, true_positives, false_positives, false_negatives

# Calculate metrics
shd = calculate_shd(ground_truth, generated)
precision, recall, f1, tp, fp, fn = calculate_metrics(ground_truth, generated)

print("CHILD Dataset Evaluation")
print("=" * 40)
print("Node order:", node_names)
print(f"Total nodes: {len(node_names)}")

print(f"\n=== Results ===")
print(f"Structured Hamming Distance: {shd}")
print(f"True Positives: {tp}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-Score: {f1:.3f}")

# Show edge differences
print(f"\n=== Edge Analysis ===")
print(f"Ground Truth Edges ({len(gt_edges)}):")
for source, target in gt_edges:
    print(f"  {source} → {target}")

print(f"\nGenerated Edges ({len(gen_edges)}):")
for source, target in gen_edges:
    print(f"  {source} → {target}")

print("\nCorrect Edges (True Positives):")
correct_edges = []
for i in range(20):
    for j in range(20):
        if ground_truth[i][j] == 1 and generated[i][j] == 1:
            correct_edges.append(f"  {node_names[i]} → {node_names[j]}")
if correct_edges:
    for edge in correct_edges:
        print(edge)
else:
    print("  None")

print("\nMissed Edges (False Negatives):")
missed_edges = []
for i in range(20):
    for j in range(20):
        if ground_truth[i][j] == 1 and generated[i][j] == 0:
            missed_edges.append(f"  {node_names[i]} → {node_names[j]}")
if missed_edges:
    for edge in missed_edges:
        print(edge)
else:
    print("  None")

print("\nIncorrect Edges (False Positives):")
incorrect_edges = []
for i in range(20):
    for j in range(20):
        if ground_truth[i][j] == 0 and generated[i][j] == 1:
            incorrect_edges.append(f"  {node_names[i]} → {node_names[j]}")
if incorrect_edges:
    for edge in incorrect_edges:
        print(edge)
else:
    print("  None")

CHILD Dataset Evaluation
Node order: ['BirthAsphyxia', 'HypDistrib', 'HypoxiaInO2', 'LowerBodyO2', 'RUQO2', 'CO2', 'CO2Report', 'ChestXray', 'XrayReport', 'Grunting', 'GruntingReport', 'Disease', 'Age', 'LVH', 'LVHreport', 'DuctFlow', 'CardiacMixing', 'LungParench', 'LungFlow', 'Sick']
Total nodes: 20

=== Results ===
Structured Hamming Distance: 21
True Positives: 13
False Positives: 9
False Negatives: 12
Precision: 0.591
Recall: 0.520
F1-Score: 0.553

=== Edge Analysis ===
Ground Truth Edges (25):
  BirthAsphyxia → Disease
  HypDistrib → LowerBodyO2
  HypoxiaInO2 → LowerBodyO2
  HypoxiaInO2 → RUQO2
  CO2 → CO2Report
  ChestXray → XrayReport
  Grunting → GruntingReport
  Disease → Age
  Disease → LVH
  Disease → DuctFlow
  Disease → CardiacMixing
  Disease → LungParench
  Disease → LungFlow
  Disease → Sick
  LVH → LVHreport
  DuctFlow → HypDistrib
  CardiacMixing → HypDistrib
  CardiacMixing → HypoxiaInO2
  LungParench → HypoxiaInO2
  LungParench → CO2
  LungParench → ChestXray
  Lun